# **ETL FACT PRODUCTS**

In [0]:
%sql
merge into
  products.gold.fact_product_prices as target
using (
  select
    md5(concat('|', dm.market_id, dp.product_id, dt.date_id)) as product_hash,
    dp.product_id,
    dm.market_id,
    dt.date_id,
    ps.start_date_price,
    ps.end_date_price,
    ps.price_currency
  from
    products.silver.product_list_silver ps
      inner join products.gold.dim_product dp
        on ps.product_name = dp.product_name
        and ps.product_unit = dp.product_unit
      inner join products.gold.dim_market dm
        on ps.market_name = dm.market_name
        and ps.department = dm.department
        and ps.city = dm.city
      inner join products.gold.dim_time dt
        on dt.start_market_date = ps.start_date_market
        and dt.end_market_date = ps.end_date_market
  order by
    dt.date_id
) as source
on 
    target.product_hash = source.product_hash
when not matched then insert (
    product_hash,
    product_id,
    market_id,
    date_id,
    start_date_price,
    end_date_price,
    price_currency
  )
  values (
    source.product_hash,
    source.product_id,
    source.market_id,
    source.date_id,
    source.start_date_price,
    source.end_date_price,
    source.price_currency
  )

In [0]:
%sql
with deactivate_fact_prices as(
    SELECT
        product_hash,
        ROW_NUMBER() OVER (PARTITION BY product_id, market_id ORDER BY date_id DESC) as rn
    FROM
        products.gold.fact_product_prices
)

update products.gold.fact_product_prices
set is_current = false
where product_hash in (
    select product_hash
    from deactivate_fact_prices
    where rn > 1
)

In [0]:
%sql
select * from products.gold.fact_product_prices

In [0]:
%sql
/*

--e9e9a3911133a2f5a6690f7c07a35a34 20260522 true

--0c73ad96942b4dde2f9457d627347089 20260515 false

-- e9e9a3911133a2f5a6690f7c07a35a34	true	1
-- 0c73ad96942b4dde2f9457d627347089	false	2

UPDATE products.gold.fact_product_prices
SET is_current = false
WHERE product_hash IN (
    SELECT product_hash
    FROM (
        SELECT 
            product_hash,
            ROW_NUMBER() OVER (
                PARTITION BY product_id, market_id 
                ORDER BY date_id DESC
            ) as rn
        FROM products.gold.fact_product_prices
    ) sub
    WHERE sub.rn > 1 -- Todos los que no sean el más reciente pasan a false
);
*/